In [42]:
! pip install numpy pandas matplotlib scipy networkx torch scikit-learn pyod seaborn
!mkdir -p Image


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
A subdirectory or file -p already exists.
Error occurred while processing: -p.


In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from scipy import signal
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from pyod.models.iforest import IForest
import time
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams.update({'font.size': 12})
os.makedirs("Images", exist_ok=True)

In [45]:
def safe_plot(func):
    """Decorator to safely handle matplotlib plotting errors"""
    def wrapper(*args, **kwargs):
        try:
            result = func(*args, **kwargs)
            plt.close('all')  # Ensure all plots are closed
            return result
        except Exception as e:
            print(f"Error in plotting function {func.__name__}: {e}")
            plt.close('all')  # Ensure all plots are closed even on error
            return None
    return wrapper

dataset_path = "Dataset\merged_data_20Hz.csv"
df = pd.read_csv(dataset_path)
print(f"Dataset shape: {df.shape}")

Dataset shape: (394370, 13)


In [46]:
@safe_plot
def quick_time_series_analysis(df):
    print("Performing time series analysis...")
    time_col = df.columns[0]
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if time_col in numeric_cols:
        numeric_cols.remove(time_col)
    
    window_size = 10
    smoothed_df = df.copy()
    
    for col in numeric_cols:
        smoothed_df[f"{col}_smoothed"] = df[col].rolling(window=window_size, center=True).mean()
        smoothed_df[f"{col}_smoothed"].fillna(df[col], inplace=True)
        smoothed_df[f"{col}_residual"] = df[col] - smoothed_df[f"{col}_smoothed"]
    
    plt.figure(figsize=(12, 6))
    plt.plot(df.index[:500], df[numeric_cols[0]][:500], label='Original')
    plt.plot(df.index[:500], smoothed_df[f"{numeric_cols[0]}_smoothed"][:500], 
             label='Smoothed', color='red')
    plt.xlabel('Time Index (Sample Number)')
    plt.ylabel(f'Signal Value ({numeric_cols[0]})')
    plt.title(f'Original vs Smoothed Signal - First 500 Points')
    plt.legend()
    plt.tight_layout()
    plt.savefig('Images/quick_smoothed_signal.png', dpi=300)
    plt.close()
    
    return smoothed_df, numeric_cols

In [47]:
def build_efficient_graph(df, feature_cols, window_size=20, k_neighbors=5):
    print("Building graph...")
    start_time = time.time()
    
    features = df[feature_cols].values
    
    G = nx.Graph()
    
    for i in range(len(df)):
        G.add_node(i, features=features[i])
    
    for i in range(len(df) - 1):
        G.add_edge(i, i + 1, weight=1.0)
    
    for i in range(0, len(df), 10):
        start_idx = max(0, i - window_size)
        end_idx = min(len(df), i + window_size)
        
        local_points = [(j, features[j]) for j in range(start_idx, end_idx) 
                        if j != i and abs(j - i) > 1]
        
        if not local_points:
            continue
            
        local_indices, local_features = zip(*local_points)
        local_features = np.array(local_features)
        
        distances = np.linalg.norm(local_features - features[i], axis=1)
        
        if len(distances) > k_neighbors:
            nearest_indices = np.argsort(distances)[:k_neighbors]
            for idx in nearest_indices:
                j = local_indices[idx]
                G.add_edge(i, j, weight=1.0)
    
    print(f"Graph built in {time.time() - start_time:.2f} seconds")
    print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
    return G

In [48]:
@safe_plot
def simple_1d_conv_detection(df, feature_cols):
    print("Performing 1D convolution anomaly detection...")
    start_time = time.time()
    
    features = df[feature_cols].values
    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
    
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    window_size = 10
    residuals = []
    
    for i in range(len(feature_cols)):
        feature = features_scaled[:, i]
        
        kernel = np.ones(window_size) / window_size
        filtered = signal.convolve(feature, kernel, mode='same')
        
        residual = feature - filtered
        residuals.append(residual)
    
    residuals = np.column_stack(residuals)
    residual_norms = np.linalg.norm(residuals, axis=1)
    
    clf = IForest(n_estimators=100, contamination=0.05, random_state=42, n_jobs=-1)
    clf.fit(residual_norms.reshape(-1, 1))
    
    anomaly_scores = clf.decision_scores_
    anomaly_predictions = clf.predict(residual_norms.reshape(-1, 1))
    
    result_df = df.copy()
    result_df['conv_anomaly_score'] = anomaly_scores
    result_df['conv_anomaly'] = anomaly_predictions
    
    plt.figure(figsize=(12, 8))
    
    plt.subplot(2, 1, 1)
    plt.plot(anomaly_scores, label='1D Conv Anomaly Score')
    anomaly_idx = np.where(anomaly_predictions == 1)[0]
    plt.scatter(anomaly_idx, anomaly_scores[anomaly_idx], color='red', label='Anomalies')
    plt.xlabel('Time Index (Sample Number)')
    plt.ylabel('Anomaly Score')
    plt.title('1D Convolution Anomaly Detection Scores')
    plt.legend()
    
    plt.subplot(2, 1, 2)
    plt.plot(df[feature_cols[0]], label=feature_cols[0], alpha=0.7)
    plt.scatter(anomaly_idx, df.iloc[anomaly_idx][feature_cols[0]], color='red', label='Anomalies')
    plt.xlabel('Time Index (Sample Number)')
    plt.ylabel(f'Signal Value ({feature_cols[0]})')
    plt.title(f'Signal with Detected Anomalies')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig('Images/conv_anomaly_detection.png', dpi=300)
    plt.close()
    
    print(f"1D convolution completed in {time.time() - start_time:.2f} seconds")
    
    return result_df, residuals

In [49]:
@safe_plot
def fast_graph_anomaly_detection(G, df, feature_cols):
    print("Performing graph-based anomaly detection...")
    start_time = time.time()
    
    x = np.array([G.nodes[i]['features'] for i in range(len(G))])
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(x)
    
    clf = IForest(n_estimators=100, contamination=0.05, random_state=42, n_jobs=-1)
    clf.fit(x_scaled)
    
    anomaly_scores = clf.decision_scores_
    anomaly_predictions = clf.predict(x_scaled)
    
    result_df = df.copy()
    result_df['graph_anomaly_score'] = anomaly_scores
    result_df['graph_anomaly'] = anomaly_predictions
    
    plt.figure(figsize=(12, 8))
    
    plt.subplot(2, 1, 1)
    plt.plot(anomaly_scores, label='Graph Anomaly Score')
    anomaly_idx = np.where(anomaly_predictions == 1)[0]
    plt.scatter(anomaly_idx, anomaly_scores[anomaly_idx], color='red', label='Anomalies')
    plt.xlabel('Time Index (Sample Number)')
    plt.ylabel('Anomaly Score')
    plt.title('Graph-based Anomaly Detection Scores')
    plt.legend()
    
    plt.subplot(2, 1, 2)
    plt.plot(df[feature_cols[0]], label=feature_cols[0], alpha=0.7)
    plt.scatter(anomaly_idx, df.iloc[anomaly_idx][feature_cols[0]], color='red', label='Anomalies')
    plt.xlabel('Time Index (Sample Number)')
    plt.ylabel(f'Signal Value ({feature_cols[0]})')
    plt.title(f'Signal with Graph-based Detected Anomalies')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig('Images/graph_anomaly_detection.png', dpi=300)
    plt.close()
    
    print(f"Graph anomaly detection completed in {time.time() - start_time:.2f} seconds")
    
    return result_df, x_scaled

In [50]:
@safe_plot
def quick_anomaly_analysis(df, result_df):
    print("Analyzing anomalies...")
    start_time = time.time()
    
    anomaly_cols = [col for col in result_df.columns if 'anomaly' in col and 'score' not in col]
    if not anomaly_cols:
        print("No anomaly columns found in result_df")
        return
        
    main_anomaly_col = anomaly_cols[0]
    anomaly_indices = result_df[main_anomaly_col] == 1
    anomaly_count = anomaly_indices.sum()
    
    print(f"Detected {anomaly_count} anomalies out of {len(result_df)} data points "
          f"({anomaly_count/len(result_df)*100:.2f}%)")
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) > 0 and df.columns[0] in numeric_cols:
        numeric_cols.remove(df.columns[0])
    
    if len(numeric_cols) > 0:
        normal_stats = df.loc[~anomaly_indices, numeric_cols].describe()
        anomaly_stats = df.loc[anomaly_indices, numeric_cols].describe()
        
        print("\nSummary statistics for normal points:")
        print(normal_stats.loc[['mean', 'std', 'min', 'max']])
        
        print("\nSummary statistics for anomalous points:")
        print(anomaly_stats.loc[['mean', 'std', 'min', 'max']])
    
    plt.figure(figsize=(12, 3))
    plt.scatter(range(len(df)), np.zeros(len(df)), 
                s=5, alpha=0.3, label='All Data Points')
    plt.scatter(np.where(anomaly_indices)[0], np.zeros(anomaly_count), 
                s=20, color='red', label='Anomalies')
    plt.yticks([])
    plt.xlabel('Time Index (Sample Number)')
    plt.title('Temporal Distribution of Anomalies')
    plt.legend()
    plt.tight_layout()
    plt.savefig('Images/anomaly_temporal_distribution.png', dpi=300)
    plt.close()
    
    print(f"Analysis completed in {time.time() - start_time:.2f} seconds")

In [51]:
@safe_plot
def improved_voltage_current_analysis(df, feature_cols, contamination=0.01):
    print("Performing voltage/current anomaly detection...")
    
    # Identify voltage and current columns
    voltage_cols = [col for col in feature_cols if 'volt' in col.lower()]
    current_cols = [col for col in feature_cols if 'curr' in col.lower() or 'amp' in col.lower()]
    
    # If no specific columns found, use first columns as proxies
    if not voltage_cols and len(feature_cols) >= 1:
        voltage_cols = [feature_cols[0]]
    if not current_cols and len(feature_cols) >= 2:
        current_cols = [feature_cols[1]]
    
    result_df = df.copy()
    
    # Process voltage and current columns
    for col_type, cols in [('voltage', voltage_cols), ('current', current_cols)]:
        if not cols:
            continue
            
        print(f"Analyzing {col_type} columns: {cols}")
        
        for col in cols:
            print(f"Processing {col}...")
            signal = df[col].values
            signal = np.nan_to_num(signal, nan=0.0)
            
            # Find transitions
            transitions = np.abs(np.diff(signal))
            threshold = np.percentile(transitions, 99)
            transition_points = np.where(transitions > threshold)[0]
            
            # Create features for anomaly detection
            window_size = 5
            gradient = np.gradient(signal)
            gradient_smoothed = np.convolve(gradient, np.ones(window_size)/window_size, mode='same')
            
            features = np.column_stack([
                signal,
                gradient,
                gradient_smoothed,
                np.roll(signal, 1),
                np.roll(signal, -1),
            ])
            
            # Remove boundary effects
            features = features[5:-5]
            
            # Standardize features
            scaler = StandardScaler()
            features_scaled = scaler.fit_transform(features)
            
            # Detect anomalies
            detector = IForest(n_estimators=100, contamination=contamination, random_state=42, n_jobs=-1)
            detector.fit(features_scaled)
            
            # Get anomaly scores and predictions
            anomaly_scores = np.zeros(len(signal))
            anomaly_scores[5:-5] = detector.decision_scores_
            
            anomaly_predictions = np.zeros(len(signal))
            predictions = detector.predict(features_scaled)
            anomaly_predictions[5:-5] = predictions
            
            # Add transition points to anomalies
            for point in transition_points:
                if point > 0 and point < len(anomaly_predictions) - 1:
                    window = 2
                    anomaly_predictions[point-window:point+window+1] = 1
            
            # Store results
            result_df[f'{col}_anomaly_score'] = anomaly_scores
            result_df[f'{col}_anomaly'] = anomaly_predictions
            
            # Plot signal with anomalies
            plt.figure(figsize=(16, 8))
            
            plt.subplot(2, 1, 1)
            plt.plot(signal, label=f'{col_type}: {col}')
            anomaly_idx = np.where(anomaly_predictions == 1)[0]
            plt.scatter(anomaly_idx, signal[anomaly_idx], color='red', s=30, label='Anomalies')
            plt.xlabel('Time Index (Sample Number)')
            plt.ylabel(f'{col_type.title()} ({col})')
            plt.title(f'{col_type.title()} Signal with Detected Anomalies')
            plt.legend()
            
            plt.subplot(2, 1, 2)
            plt.plot(anomaly_scores, label='Anomaly Score')
            plt.axhline(y=detector.threshold_, color='r', linestyle='--', label='Threshold')
            plt.scatter(anomaly_idx, anomaly_scores[anomaly_idx], color='red', s=30)
            plt.xlabel('Time Index (Sample Number)')
            plt.ylabel('Anomaly Score')
            plt.title(f'Anomaly Scores for {col_type.title()} Signal')
            plt.legend()
            
            plt.tight_layout()
            plt.savefig(f'Images/{col}_anomalies.png', dpi=300)
            plt.close()
            
            # Plot transitions
            plt.figure(figsize=(16, 6))
            plt.plot(signal, label=f'{col_type}: {col}')
            for point in transition_points:
                plt.axvline(x=point, color='g', alpha=0.3)
            plt.scatter(anomaly_idx, signal[anomaly_idx], color='red', s=30, label='Anomalies')
            plt.xlabel('Time Index (Sample Number)')
            plt.ylabel(f'{col_type.title()} ({col})')
            plt.title(f'{col_type.title()} Signal with Transitions and Anomalies')
            plt.legend()
            plt.tight_layout()
            plt.savefig(f'Images/{col}_transitions.png', dpi=300)
            plt.close()
    
    # Create combined anomaly flag
    anomaly_cols = [col for col in result_df.columns if 'anomaly' in col and 'score' not in col]
    if anomaly_cols:
        result_df['combined_anomaly'] = result_df[anomaly_cols].max(axis=1)
    
    print("Voltage/current anomaly detection completed")
    return result_df

In [52]:
@safe_plot
def plot_voltage_current_phase_portrait(df, feature_cols):
    print("Creating voltage-current phase portrait...")
    
    voltage_cols = [col for col in feature_cols if 'volt' in col.lower()]
    current_cols = [col for col in feature_cols if 'curr' in col.lower() or 'amp' in col.lower()]
    
    if not voltage_cols and len(feature_cols) >= 1:
        voltage_cols = [feature_cols[0]]
    if not current_cols and len(feature_cols) >= 2:
        current_cols = [feature_cols[1]]
    
    if not voltage_cols or not current_cols:
        print("No voltage or current columns found")
        return
    
    v_col = voltage_cols[0]
    c_col = current_cols[0]
    
    # Get unique voltage-current pairs with counts
    value_counts = df.groupby([v_col, c_col]).size().reset_index(name='count')
    
    # Plot the phase portrait with point size proportional to frequency
    plt.figure(figsize=(14, 10))
    
    scatter = plt.scatter(value_counts[v_col], value_counts[c_col], 
                s=value_counts['count']*2, alpha=0.6, c=value_counts['count'], 
                cmap='viridis')
    
    # Add colorbar to indicate frequency
    plt.colorbar(scatter, label='Frequency (Number of Occurrences)')
    
    # Add labels
    for i, row in value_counts.iterrows():
        if row['count'] > max(value_counts['count'])/10:  # Only label major points
            plt.text(row[v_col], row[c_col], f"n={row['count']}", 
                     fontsize=10, ha='center', va='bottom')
    
    plt.grid(True)
    plt.xlabel(f'Voltage ({v_col}) - Volts', fontsize=14)
    plt.ylabel(f'Current ({c_col}) - Amperes', fontsize=14)
    plt.title('Voltage-Current Phase Portrait - Operating Points', fontsize=16)
    
    plt.tight_layout()
    plt.savefig('Images/voltage_current_phase_portrait.png', dpi=300)
    plt.close()
    
    print("Voltage-current phase portrait created")

In [53]:
@safe_plot
def feature_heatmap(df, feature_cols, max_features=8):
    print("Generating feature correlation heatmap...")
    
    # Select a subset of features if too many
    if len(feature_cols) > max_features:
        features_to_use = feature_cols[:max_features]
    else:
        features_to_use = feature_cols
    
    # Calculate correlation matrix
    corr_matrix = df[features_to_use].corr()
    
    # Create heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, 
                linewidths=0.5, fmt='.2f')
    plt.title('Feature Correlation Heatmap', fontsize=16)
    plt.tight_layout()
    plt.savefig('Images/feature_correlation_heatmap.png', dpi=300)
    plt.close()
    
    # Create pairplot for the main features (if not too many)
    if len(features_to_use) <= 4:  # Limit to 4 features to keep pairplot manageable
        try:
            g = sns.pairplot(df[features_to_use], diag_kind='kde')
            g.fig.suptitle('Feature Distributions and Relationships', y=1.02, fontsize=16)
            plt.savefig('Images/feature_pairplot.png', dpi=300)
            plt.close()
        except Exception as e:
            print(f"Error creating pairplot: {e}")
    
    print("Feature heatmap and relationships visualized")

In [54]:
def main():
    print("Starting enhanced time series analysis and anomaly detection...")
    print(f"All visualizations will be saved in the 'Images' folder")
    total_start_time = time.time()
    
    try:
        df = pd.read_csv(dataset_path)
        print(f"Loaded dataset with shape: {df.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return
    
    try:
        smoothed_df, feature_cols = quick_time_series_analysis(df)
        
        G = build_efficient_graph(df, feature_cols, window_size=20, k_neighbors=3)
        
        result_df, features = simple_1d_conv_detection(df, feature_cols)
        
        # Use try/except for each major step
        try:
            result_df, graph_features = fast_graph_anomaly_detection(G, df, feature_cols)
        except Exception as e:
            print(f"Error in graph analysis: {e}")
        
        try:
            result_df = improved_voltage_current_analysis(df, feature_cols, contamination=0.005)
        except Exception as e:
            print(f"Error in voltage/current analysis: {e}")
        
        try:
            plot_voltage_current_phase_portrait(df, feature_cols)
        except Exception as e:
            print(f"Error in phase portrait: {e}")
        
        quick_anomaly_analysis(df, result_df)
        
        feature_heatmap(df, feature_cols)
        
        total_time = time.time() - total_start_time
        print(f"All analysis completed in {total_time:.2f} seconds")
        print("All visualizations have been saved in the 'Images' folder")
        
    except Exception as e:
        print(f"Unexpected error: {e}")

if __name__ == "__main__":
    main()

Starting enhanced time series analysis and anomaly detection...
All visualizations will be saved in the 'Images' folder
Loaded dataset with shape: (394370, 13)
Performing time series analysis...
Building graph...
Graph built in 11.92 seconds
Graph has 394370 nodes and 512680 edges
Performing 1D convolution anomaly detection...
1D convolution completed in 4.10 seconds
Performing graph-based anomaly detection...
Graph anomaly detection completed in 4.65 seconds
Performing voltage/current anomaly detection...
Analyzing voltage columns: ['Polling of A']
Processing Polling of A...
Analyzing current columns: ['Polling of B']
Processing Polling of B...
Voltage/current anomaly detection completed
Creating voltage-current phase portrait...
Voltage-current phase portrait created
Analyzing anomalies...
Detected 30 anomalies out of 394370 data points (0.01%)

Summary statistics for normal points:
      Polling of A  Polling of B
mean     48.294568     48.058414
std      38.249008     38.175946
min